# Text Splitters — Recursive, Token & Semantic Chunking

Break documents into LLM-friendly chunks that preserve meaning — the bridge between loading and embedding.

---

In [ ]:
!pip install -q langchain langchain-text-splitters langchain-experimental langchain-openai tiktoken

In [ ]:
# ── Sample document we'll use throughout this tutorial ──
# A multi-topic text to demonstrate how different splitters handle it

SAMPLE_TEXT = """
The Transformer Architecture

The Transformer was introduced in the 2017 paper 'Attention Is All You Need' by Vaswani et al. It replaced recurrent and convolutional layers with self-attention mechanisms, enabling massive parallelization during training. The architecture consists of an encoder and decoder, each built from stacked layers of multi-head self-attention and feed-forward networks.

Self-attention allows every token in a sequence to attend to every other token, producing context-aware representations. Multi-head attention runs this process across multiple representation subspaces in parallel, capturing different types of relationships between tokens.

Applications of Transformers

Transformers power virtually every modern language model. GPT uses only the decoder stack for autoregressive text generation. BERT uses only the encoder stack for bidirectional understanding. T5 uses both encoder and decoder for sequence-to-sequence tasks.

Beyond NLP, transformers have been adapted for computer vision (Vision Transformer / ViT), protein structure prediction (AlphaFold), audio generation (Jukebox), and robotic control. The architecture's flexibility and scalability make it the dominant paradigm in modern AI.

Training and Optimization

Training large transformers requires significant compute resources. Key techniques include mixed-precision training (FP16/BF16), gradient checkpointing to reduce memory usage, and distributed training across multiple GPUs or TPUs. Learning rate scheduling with warmup is critical for stable convergence.

Fine-tuning adapts a pretrained transformer to specific downstream tasks. Parameter-efficient methods like LoRA and QLoRA modify only a small subset of weights, dramatically reducing memory and compute requirements while maintaining performance close to full fine-tuning.
""".strip()

print(f"Document length: {len(SAMPLE_TEXT)} characters")
print(f"Paragraphs: {SAMPLE_TEXT.count(chr(10)+chr(10)) + 1}")

---

## 1 · Why Split Text?

**The Problem** — LLMs have token limits. A 100-page PDF won't fit in a single prompt. Even if it did, more tokens = higher cost and diluted attention.

**The Solution** — Split documents into smaller chunks. Each chunk gets embedded independently, and only relevant chunks are retrieved at query time.

> Splitting is like cutting a textbook into index cards. You don't hand someone the entire book when they ask a question — you hand them the 3 most relevant cards.

In [ ]:
# ── The naive approach: just slice every N characters ──
# This is BAD — watch what happens

chunk_size = 200
naive_chunks = [SAMPLE_TEXT[i:i+chunk_size] for i in range(0, len(SAMPLE_TEXT), chunk_size)]

print(f"Naive splitting into {len(naive_chunks)} chunks of {chunk_size} chars:\n")
for i, chunk in enumerate(naive_chunks[:3]):
    print(f"--- Chunk {i} ---")
    print(repr(chunk))    # repr() shows \n characters explicitly
    print()

# Notice: words are cut mid-sentence, paragraphs are split arbitrarily

### What Just Happened?

Naive slicing cuts through sentences and words. "self-attention mecha" / "nisms" — that's useless for retrieval. LangChain's text splitters solve this by respecting natural text boundaries.

---

## 2 · RecursiveCharacterTextSplitter — The Default

**The Problem** — Naive splitting destroys meaning by cutting mid-sentence.

**The Solution** — Split on natural boundaries in order: paragraphs → newlines → sentences → spaces → characters. Only fall back when larger separators produce oversized chunks.

> Like a careful editor who first tries to break at chapter boundaries, then paragraphs, then sentences — always preferring the cleanest cut.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Basic recursive splitting ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,                                    # max characters per chunk
    chunk_overlap=30,                                  # shared chars between consecutive chunks
    separators=["\n\n", "\n", ". ", " ", ""],        # default hierarchy (explicit here for clarity)
    length_function=len,                               # measure by character count
)

chunks = splitter.split_text(SAMPLE_TEXT)

print(f"Split into {len(chunks)} chunks (max {300} chars each)\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

In [ ]:
# ── Using split_documents() with Document objects (real-world pattern) ──
from langchain_core.documents import Document

# Simulate what a document loader would produce
docs = [
    Document(page_content=SAMPLE_TEXT, metadata={"source": "transformers.pdf", "page": 0})
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)       # preserves metadata on every chunk

print(f"Documents in: {len(docs)} → Chunks out: {len(chunks)}\n")
for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i}: {len(chunk.page_content)} chars | metadata: {chunk.metadata}")

### What Just Happened?

- The splitter tried `\n\n` (paragraph breaks) first — most chunks split cleanly at paragraph boundaries
- For paragraphs longer than 300 chars, it fell back to `\n`, then `. `, then ` `
- `split_documents()` preserves the original metadata on every chunk — each chunk still knows its source file and page number
- The 30-char overlap means the last sentence of chunk N appears at the start of chunk N+1

> **When to use:** The recommended default for most RAG applications. Start here and only move to more advanced strategies if retrieval metrics demand it.

---

## 3 · chunk_size and chunk_overlap — Tuning the Split

**The Problem** — Too large = wasted context space. Too small = lost context. How do you choose?

**The Solution** — Experiment. There's no universal best value — it depends on your embedding model, retrieval strategy, and document type.

> `chunk_size` is how big each index card is. `chunk_overlap` is copying the last sentence of one card onto the next — so ideas that span a boundary don't get lost.

In [ ]:
# ── Compare different chunk_size values on the same text ──

for size in [100, 300, 500, 1000]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.1),   # 10% overlap — a good starting point
    )
    chunks = splitter.split_text(SAMPLE_TEXT)
    avg_len = sum(len(c) for c in chunks) / len(chunks)
    print(f"chunk_size={size:>5} | overlap={int(size*0.1):>3} | chunks={len(chunks):>2} | avg_len={avg_len:.0f} chars")

In [ ]:
# ── Visualize overlap between consecutive chunks ──

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
chunks = splitter.split_text(SAMPLE_TEXT)

# Show the overlap region between chunk 0 and chunk 1
print("=== Chunk 0 (end) ===")
print(f"...{chunks[0][-60:]}")
print()
print("=== Chunk 1 (start) ===")
print(f"{chunks[1][:60]}...")
print()

# Find the overlapping text
for overlap_len in range(min(len(chunks[0]), len(chunks[1])), 0, -1):
    if chunks[0].endswith(chunks[1][:overlap_len]):
        print(f"Overlap found: {overlap_len} chars")
        print(f"Shared text: '{chunks[1][:overlap_len]}'")
        break

### What Just Happened?

- Smaller `chunk_size` → more chunks, each with less context but more specific retrieval
- Larger `chunk_size` → fewer chunks, more context per chunk but less precise retrieval
- `chunk_overlap` creates a shared region between consecutive chunks, preventing context loss at boundaries
- **Rule of thumb:** overlap = 10-20% of chunk_size. Start at 10% and increase if retrieval misses boundary-spanning ideas

---

## 4 · TokenTextSplitter — Chunk by Token Count

**The Problem** — LLMs count tokens, not characters. 500 characters might be 100 tokens or 200 tokens depending on the text.

**The Solution** — `TokenTextSplitter` uses the model's actual tokenizer (tiktoken) to measure chunk size in tokens.

> If characters are letters and tokens are words, `RecursiveCharacterTextSplitter` measures by letter count while `TokenTextSplitter` measures by word count. The word count is what the LLM actually cares about.

In [ ]:
from langchain_text_splitters import TokenTextSplitter

# ── Split by token count instead of character count ──
splitter = TokenTextSplitter(
    chunk_size=100,                    # max TOKENS per chunk (not characters)
    chunk_overlap=10,                  # token overlap
    encoding_name="cl100k_base"        # tokenizer used by GPT-4o, GPT-4o-mini
)

chunks = splitter.split_text(SAMPLE_TEXT)

print(f"Split into {len(chunks)} chunks (max 100 tokens each)\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:150] + ("..." if len(chunk) > 150 else ""))
    print()

In [ ]:
# ── Verify token counts with tiktoken ──
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

print("Token counts per chunk:")
for i, chunk in enumerate(chunks):
    token_count = len(enc.encode(chunk))
    char_count = len(chunk)
    print(f"  Chunk {i}: {token_count:>4} tokens | {char_count:>4} chars | ratio: {char_count/token_count:.1f} chars/token")

In [ ]:
# ── Compare: same text, character vs token splitting ──

char_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
token_splitter = TokenTextSplitter(chunk_size=100, chunk_overlap=10)

char_chunks = char_splitter.split_text(SAMPLE_TEXT)
token_chunks = token_splitter.split_text(SAMPLE_TEXT)

print(f"Character splitter (400 chars): {len(char_chunks)} chunks")
print(f"Token splitter (100 tokens):    {len(token_chunks)} chunks")
print(f"\n(~400 chars ≈ ~100 tokens for English text, so these are roughly equivalent)")

### What Just Happened?

- `TokenTextSplitter` splits based on actual token count using tiktoken
- For English text, ~4 characters ≈ 1 token — so 100 tokens ≈ 400 characters
- Token-based splitting ensures you never exceed model limits, regardless of text content
- The `encoding_name` parameter must match your target model's tokenizer

> **When to use:** When you need precise token budget control — fitting chunks into a fixed context window or managing API costs per request.

---

## 5 · SemanticChunker — Meaning-Aware Splitting

**The Problem** — Character and token splitters are blind to meaning. A paragraph about two topics stays in one chunk. A coherent idea gets split across two.

**The Solution** — `SemanticChunker` embeds every sentence, compares adjacent sentence similarities, and cuts where the meaning shifts.

> Instead of cutting a book every 500 words, you read it and cut at the point where the author changes topics — regardless of word count.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# ── Semantic chunking — splits based on meaning, not size ──
splitter = SemanticChunker(
    embeddings=OpenAIEmbeddings(),                # embedding model to measure similarity
    breakpoint_threshold_type="percentile",       # method to detect topic shifts
    breakpoint_threshold_amount=85,               # split at 85th percentile of similarity drops
)

chunks = splitter.create_documents([SAMPLE_TEXT])

print(f"Semantic chunking produced {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200] + ("..." if len(chunk.page_content) > 200 else ""))
    print()

In [ ]:
# ── Compare breakpoint threshold types ──

threshold_types = ["percentile", "standard_deviation", "interquartile"]

for t_type in threshold_types:
    splitter = SemanticChunker(
        embeddings=OpenAIEmbeddings(),
        breakpoint_threshold_type=t_type,
    )
    chunks = splitter.create_documents([SAMPLE_TEXT])
    sizes = [len(c.page_content) for c in chunks]
    print(f"{t_type:>22s} | chunks: {len(chunks):>2} | avg size: {sum(sizes)//len(sizes):>4} chars | range: {min(sizes)}-{max(sizes)}")

### What Just Happened?

- `SemanticChunker` embedded each sentence and compared adjacent pairs
- Where similarity dropped significantly (topic shift), it created a breakpoint
- Chunk sizes are **unpredictable** — they're driven by meaning, not a fixed size limit
- Different `breakpoint_threshold_type` values produce different granularity

**Breakpoint types:**
- `percentile` — splits at the Nth percentile of similarity drops (most intuitive)
- `standard_deviation` — splits when similarity drops more than N standard deviations from the mean
- `interquartile` — uses IQR-based outlier detection to find breakpoints
- `gradient` — looks for large changes in the rate of similarity change

> **When to use:** Multi-topic documents without clear structural markers. Tradeoff: requires embedding API calls per sentence — adds latency and cost. Benchmark against `RecursiveCharacterTextSplitter` before committing.

---

## 6 · Side-by-Side Comparison

Same document, three different splitters — compare the results.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# ── Three splitters, same text ──

recursive = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
token = TokenTextSplitter(chunk_size=100, chunk_overlap=10)
semantic = SemanticChunker(embeddings=OpenAIEmbeddings(), breakpoint_threshold_type="percentile")

r_chunks = recursive.split_text(SAMPLE_TEXT)
t_chunks = token.split_text(SAMPLE_TEXT)
s_chunks = semantic.create_documents([SAMPLE_TEXT])

print("=" * 60)
print(f"{'Splitter':<25} {'Chunks':<10} {'Avg Size':<12} {'Min':<8} {'Max':<8}")
print("=" * 60)

for name, chunks_list in [
    ("Recursive (400 chars)", r_chunks),
    ("Token (100 tokens)", t_chunks),
    ("Semantic (percentile)", [c.page_content for c in s_chunks]),
]:
    sizes = [len(c) for c in chunks_list]
    print(f"{name:<25} {len(sizes):<10} {sum(sizes)//len(sizes):<12} {min(sizes):<8} {max(sizes):<8}")

print("\n" + "=" * 60)
print("Key observation: Recursive and Token produce predictable sizes.")
print("Semantic produces variable sizes driven by topic boundaries.")

---

## 7 · Full Pipeline — Load → Split → (Embed)

Connecting Tutorial 04 (loaders) with Tutorial 05 (splitters) — the standard two-step pattern before embedding.

In [ ]:
# ── Create a sample PDF (same approach as Tutorial 04) ──
try:
    from fpdf import FPDF
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "fpdf2"])
    from fpdf import FPDF

pdf = FPDF()
for title, content in [
    ("Attention Mechanisms", "Self-attention computes a weighted sum of all positions in a sequence. Each token attends to every other token, producing context-aware representations. Multi-head attention runs this in parallel across multiple subspaces."),
    ("Transfer Learning", "Pre-trained models like GPT and BERT learn general language representations from massive corpora. Fine-tuning adapts these models to specific tasks with much less data and compute than training from scratch."),
    ("Retrieval-Augmented Generation", "RAG combines retrieval with generation. Instead of relying solely on parametric knowledge, the model retrieves relevant documents from a knowledge base and uses them as context for generating responses."),
]:
    pdf.add_page()
    pdf.set_font("Helvetica", size=16)
    pdf.cell(text=title, new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", size=12)
    pdf.multi_cell(w=0, text=content)

pdf.output("ml_concepts.pdf")
print("Created: ml_concepts.pdf (3 pages)")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Step 1: Load ──
loader = PyPDFLoader("ml_concepts.pdf")
docs = loader.load()
print(f"Step 1 — Loaded: {len(docs)} pages\n")

for doc in docs:
    print(f"  Page {doc.metadata['page']}: {len(doc.page_content)} chars")

# ── Step 2: Split ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
)
chunks = splitter.split_documents(docs)           # metadata propagates to every chunk
print(f"\nStep 2 — Split: {len(docs)} pages → {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
    print(f"  Chunk {i} (page {chunk.metadata['page']}): {len(chunk.page_content)} chars")
    print(f"    '{chunk.page_content[:80]}...'\n")

# ── Step 3: Embed (preview — covered in RAG tutorials) ──
print("Step 3 — Next: embed these chunks and store in a vector database.")
print("         Covered in Tutorial 06 (RAG + FAISS).")

### What Just Happened?

This is the standard RAG ingestion pattern: **load → split → embed → store**. We completed the first two steps. Each chunk retains its source metadata, so when a chunk is retrieved later, you know exactly which page and file it came from.

---

## Key Takeaways

| Splitter | Measures By | Best For | Tradeoff |
|----------|------------|----------|----------|
| RecursiveCharacter | Characters | Default for most RAG | Fast, no API calls |
| TokenTextSplitter | Tokens (tiktoken) | Precise token budgets | Slightly slower |
| SemanticChunker | Embedding similarity | Multi-topic documents | Needs embedding calls |

**Practical defaults:**
- Start with `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`
- Overlap = 10-20% of chunk_size
- Only move to SemanticChunker if retrieval benchmarks justify the added cost

**Next →** [06 · RAG + FAISS](../06-rag-faiss/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*